# Granos de café — entrenamiento en Google Colab

Entrena y compara los dos enfoques con el dataset re-exportado de Label Studio:

- **A. Una etapa**: YOLO de 14 clases (línea base).
- **B. Dos etapas**: detector de "grano" + clasificador de defectos.

**Antes de empezar:**
1. *Entorno de ejecución → Cambiar tipo de entorno de ejecución → **GPU** (T4 o mejor)*.
2. Sube a Google Drive el dataset preparado con `preparar_export.py` comprimido en zip,
   por ejemplo `MiDrive/granos/dataset_v2.zip`. El zip debe contener `images/`, `labels/` y `data.yaml`.

Los pesos y resultados se guardan en `MiDrive/granos/`, así no se pierden si Colab se desconecta.

## 1. Configuración

In [ ]:
# --- Ajusta estas rutas ---
DRIVE_DIR   = '/content/drive/MyDrive/granos'          # carpeta de trabajo en Drive
DATASET_ZIP = f'{DRIVE_DIR}/dataset_v2.zip'            # dataset preparado con preparar_export.py
RAMA        = 'claude/yolo-grain-detection-kqf5bz'
REPO        = 'https://github.com/luisfcollazos/luisfcollazos.github.io.git'

IMGSZ_DET = 1280   # resolución para los detectores
VENTANA   = 128    # recorte fijo (px) del clasificador

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
RUNS = f'{DRIVE_DIR}/runs'     # pesos y gráficas de los entrenamientos (persisten en Drive)

## 2. Código y dependencias

Clona los scripts desde GitHub. Si el repositorio es privado, sube la carpeta `yolo-granos/`
a Drive y usa la segunda opción de la celda.

In [ ]:
%cd /content
!rm -rf repo && git clone -q --depth 1 -b {RAMA} {REPO} repo
# Alternativa si el repo es privado:  !cp -r {DRIVE_DIR}/yolo-granos /content/repo/
%cd /content/repo/yolo-granos
!pip -q install -r requirements.txt
import ultralytics; ultralytics.checks()

## 3. Dataset

Se copia del zip de Drive al disco local de Colab, que es mucho más rápido que leer de Drive durante el
entrenamiento. Luego se corrige `path` en el `data.yaml`, porque el original apunta a una carpeta de tu computador.

In [ ]:
!rm -rf /content/dataset_v2 && mkdir -p /content/dataset_v2
!unzip -q "{DATASET_ZIP}" -d /content/dataset_v2
# si el zip traía una carpeta dentro, súbela un nivel
import glob, shutil, yaml
if not os.path.exists('/content/dataset_v2/data.yaml'):
    interno = os.path.dirname(glob.glob('/content/dataset_v2/*/data.yaml')[0])
    for x in os.listdir(interno): shutil.move(os.path.join(interno, x), '/content/dataset_v2')

DATA = '/content/dataset_v2/data.yaml'
d = yaml.safe_load(open(DATA))
d['path'] = '/content/dataset_v2'
yaml.safe_dump(d, open(DATA, 'w'), allow_unicode=True, sort_keys=False)
print(open(DATA).read())
!python limpiar_caches.py --dataset /content/dataset_v2 --borrar

### Verificación rápida (opcional)

In [ ]:
!python 0_auditar_dataset.py --data {DATA} --grande sanog --pequeno sanop --no-medir --out {DRIVE_DIR}/auditoria

## 4. A — Una etapa (14 clases)

Con `--batch -1`, Ultralytics elige el batch según la memoria de la GPU (en una T4, a 1280 px, suele quedar en 4–8).
150 épocas tardan aproximadamente 1,5–3 h en una T4.

In [ ]:
!python 3_entrenar.py completo --data {DATA} --imgsz {IMGSZ_DET} --batch -1 --project {RUNS}/detect

## 5. B — Dos etapas

In [ ]:
# recortes para el clasificador + data.yaml de una sola clase para el detector
!python 2_recortar_granos.py --data {DATA} --out /content/dataset_cls --ventana {VENTANA} \
    --balancear 600 --max-por-clase 3000 --yaml-detector /content/data_1clase.yaml

In [ ]:
!python 3_entrenar.py detector --data /content/data_1clase.yaml --imgsz {IMGSZ_DET} --batch -1 --project {RUNS}/detect

In [ ]:
!python 3_entrenar.py clasificador --data /content/dataset_cls --batch 64 --project {RUNS}/classify

## 6. Comparación en el split de test

Ninguno de los dos modelos usó el split de test para elegir pesos.
Si un nombre de carpeta cambió (por ejemplo `det_granos2`, porque entrenaste dos veces), ajústalo aquí.

In [ ]:
COMPLETO = f'{RUNS}/detect/completo_granos/weights/best.pt'
DET      = f'{RUNS}/detect/det_granos/weights/best.pt'
CLS      = f'{RUNS}/classify/cls_granos/weights/best.pt'

!python 1_diagnostico_val.py --model {COMPLETO} --data {DATA} --split test --imgsz {IMGSZ_DET}

In [ ]:
!python 4_dos_etapas.py evaluar --det {DET} --cls {CLS} --data {DATA} --split test \
    --imgsz-det {IMGSZ_DET} --ventana {VENTANA} --out {DRIVE_DIR}/resultados_dos_etapas

In [ ]:
# guardar en Drive las matrices del diagnóstico de una etapa
!cp -r runs/detect/diag_* {DRIVE_DIR}/ 2>/dev/null; ls {DRIVE_DIR}

## Si Colab se desconecta

Los pesos quedan en Drive. Vuelve a ejecutar las secciones 1–3 y luego retoma el entrenamiento desde su `last.pt`:

In [ ]:
# ejemplo: retomar el modelo de una etapa
!python 3_entrenar.py completo --resume {RUNS}/detect/completo_granos/weights/last.pt